In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/adaptive-lora/results', exist_ok=True)
print("Drive ready")

Mounted at /content/drive
Drive ready


In [3]:
%%capture
!pip install transformers peft accelerate bitsandbytes datasets wandb trl


In [4]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from datasets import load_dataset

print("Imports done")

Imports done


In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Loaded. Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [6]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


In [24]:
dataset = load_dataset("fancyzhx/ag_news")

def format_example(example):
    label_names = ['World', 'Sports', 'Business', 'Sci/Tech']
    label = label_names[example['label']]
    text = f"""Classify the following news article into one of these categories: World, Sports, Business, Sci/Tech.

Article: {example['text']}

Category: {label}"""
    return {"text": text}

formatted = dataset['train'].map(format_example)
print(formatted[0]['text'])

In [ ]:
MAX_LENGTH = 256

def tokenize(example):
    result = tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

warmup_data = formatted.select(range(1000))
tokenized = warmup_data.map(tokenize, remove_columns=warmup_data.column_names)
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Tokenized {len(tokenized)} examples")

In [26]:
from torch.utils.data import DataLoader

warmup_loader = DataLoader(tokenized, batch_size=4, shuffle=True)
print(f"Batches: {len(warmup_loader)}")

Batches: 250


In [27]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4
)

sensitivity = {}
for name, param in model.named_parameters():
    if param.requires_grad:
        sensitivity[name] = 0.0

NUM_WARMUP_STEPS = 200

model.train()
step = 0

for batch in tqdm(warmup_loader):
    if step >= NUM_WARMUP_STEPS:
        break

    batch = {k: v.to(model.device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()

    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            sensitivity[name] += param.grad.norm().item()

    optimizer.step()
    optimizer.zero_grad()
    step += 1

    if step % 50 == 0:
        print(f"Step {step}/{NUM_WARMUP_STEPS} | Loss: {loss.item():.4f}")

for name in sensitivity:
    sensitivity[name] /= NUM_WARMUP_STEPS

print("Done")

  0%|          | 0/250 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
 20%|██        | 50/250 [03:27<13:36,  4.08s/it]

Step 50/200 | Loss: 0.4576


 40%|████      | 100/250 [06:54<10:28,  4.19s/it]

Step 100/200 | Loss: 0.7301


 60%|██████    | 150/250 [10:22<06:54,  4.15s/it]

Step 150/200 | Loss: 0.4777


 80%|████████  | 200/250 [13:49<03:27,  4.15s/it]

Step 200/200 | Loss: 0.6138
Done


In [32]:
save_path = '/content/drive/MyDrive/adaptive-lora/results/sensitivity_scores.json'
with open(save_path, 'w') as f:
    json.dump(sensitivity, f, indent=2)

print(f"Saved to {save_path}")

sorted_layers = sorted(sensitivity.items(), key=lambda x: x[1], reverse=True)
print("\nTop 10 most sensitive:")
for name, score in sorted_layers[:10]:
    print(f"{score:.4f}  {name}")

print("\nBottom 10 least sensitive:")
for name, score in sorted_layers[-10:]:
    print(f"{score:.4f}  {name}")

Saved to /content/drive/MyDrive/adaptive-lora/results/sensitivity_scores.json

Top 10 most sensitive:
0.5387  base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
0.4126  base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight
0.3521  base_model.model.model.layers.2.self_attn.v_proj.lora_B.default.weight
0.2983  base_model.model.model.layers.4.self_attn.v_proj.lora_B.default.weight
0.2969  base_model.model.model.layers.3.self_attn.v_proj.lora_B.default.weight
0.2647  base_model.model.model.layers.5.self_attn.v_proj.lora_B.default.weight
0.2315  base_model.model.model.layers.10.self_attn.v_proj.lora_B.default.weight
0.2208  base_model.model.model.layers.12.self_attn.v_proj.lora_B.default.weight
0.2101  base_model.model.model.layers.6.self_attn.v_proj.lora_B.default.weight
0.2094  base_model.model.model.layers.9.self_attn.v_proj.lora_B.default.weight

Bottom 10 least sensitive:
0.0124  base_model.model.model.layers.26.self_attn.v_proj.lora_A.default.weigh

In [31]:
print("hello world")


hello world


In [34]:
import json
import re

with open('sensitivity_scores.json', 'r') as f:
    raw_sensitivity = json.load(f)

# Aggregate lora_A + lora_B per layer-projection
aggregated = {}

for full_name, score in raw_sensitivity.items():
    # extract layer number and projection type (q_proj or v_proj)
    match = re.search(r'layers\.(\d+)\.self_attn\.(q_proj|v_proj)', full_name)
    if match:
        layer_num = match.group(1)
        proj_type = match.group(2)
        key = f"layer_{layer_num}_{proj_type}"

        if key not in aggregated:
            aggregated[key] = 0.0
        aggregated[key] += score   # sum A and B contributions

print(f"Aggregated into {len(aggregated)} layer-projection entries")
print(f"Expected: 64 (32 layers x 2 projections)")

# Save aggregated version
with open('sensitivity_aggregated.json', 'w') as f:
    json.dump(aggregated, f, indent=2)

# Quick check
sorted_agg = sorted(aggregated.items(), key=lambda x: x[1], reverse=True)
print("\nTop 10 aggregated:")
for name, score in sorted_agg[:10]:
    print(f"{score:.4f}  {name}")

Aggregated into 64 layer-projection entries
Expected: 64 (32 layers x 2 projections)

Top 10 aggregated:
0.6044  layer_0_v_proj
0.4644  layer_1_v_proj
0.3930  layer_2_v_proj
0.3401  layer_3_v_proj
0.3354  layer_4_v_proj
0.2972  layer_5_v_proj
0.2561  layer_10_v_proj
0.2476  layer_12_v_proj
0.2360  layer_6_v_proj
0.2308  layer_9_v_proj
